In [2]:
# Library imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path
from collections import Counter
from collections import defaultdict
from scipy.stats import wilcoxon
from scipy import stats

# Visualization settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display the dataframe to fit nicely on the screen
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)

In [3]:
# Path to the experiments directory
dir_experiments = "../experiments/nsuperpixels_filterReduction/results"

# Dicionários separados
results_files = defaultdict(dict)
superpixel_images = defaultdict(dict)

# Traverse the experiments directory
for superpixel_folder in sorted(os.listdir(dir_experiments)):
    superpixel_path = os.path.join(dir_experiments, superpixel_folder)
    if os.path.isdir(superpixel_path) and superpixel_folder.startswith("super"):
        # Armazena arquivos de resultados
        for file in sorted(os.listdir(superpixel_path)):
            if file.endswith("_results.csv") or file.endswith("_layer3_test1-classified-images.cvs"):
                results_files[superpixel_folder][file] = os.path.join(superpixel_path, file)
        # Armazena imagens dos superpixels
        for seed_folder in sorted(os.listdir(superpixel_path)):
            seed_path = os.path.join(superpixel_path, seed_folder)
            if os.path.isdir(seed_path) and seed_folder.startswith("superpixels_seed"):
                superpixel_images[superpixel_folder][seed_folder] = []
                for img_file in sorted(os.listdir(seed_path)):
                    superpixel_images[superpixel_folder][seed_folder].append(os.path.join(seed_path, img_file))
                    
# Create a structured DataFrame to store the results
results_data = []
superpixels_values = []

for superpixel, contents in results_files.items():
    # print(f"{superpixel}: {len(contents)} files")
    superpixels_values.append(int(superpixel.replace('super', '').replace('_filterReduction', '')))
    for seed, results_file in contents.items():
        if isinstance(results_file, str) and results_file.endswith('_results.csv'):
            seed_value = int(seed.replace('_results.csv', '').replace('seed', ''))
            with open(results_file, 'r') as f:
                lines = f.readlines()
                if len(lines) >= 4:
                    class1_accuracy, class2_accuracy = map(float, lines[0].strip().split(';')[:2])
                    kappa, global_accuracy = map(float, lines[1].strip().split(';')[:2])
                    nfeat = int(lines[3].strip().split(': ')[1])
                    results_data.append({
                        'superpixel': int(superpixel.replace('super', '').replace('_filterReduction', '')),
                        'filterReduction': 'filterReduction' in superpixel,
                        'seed': seed_value,
                        'class1_accuracy': class1_accuracy,
                        'class2_accuracy': class2_accuracy,
                        'kappa': kappa,
                        'global_accuracy': global_accuracy,
                        'nfeat': nfeat
                    })

results_data = sorted(results_data, key=lambda x: x['superpixel'])

# Convert the results data into a DataFrame
df_nsuper = pd.DataFrame(results_data) 

# Save the DataFrame to a CSV file
df_nsuper.to_csv('nsuperpixelsFilterReduction_results_summary.csv', index=False)

In [4]:
df_nsuper.head(100)

,superpixel,filterReduction,seed,class1_accuracy,class2_accuracy,kappa,global_accuracy,nfeat
0,50,False,1011,0.892377,0.987614,0.888503,0.975526,52900
1,50,False,1213,0.843049,0.986310,0.852222,0.968127,52900
2,50,False,123,0.856502,0.981747,0.844723,0.965851,52900
3,50,False,2735,0.852018,0.981095,0.839548,0.964713,52900
4,50,False,42,0.865471,0.983703,0.857388,0.968697,52900
...,...,...,...,...,...,...,...,...
75,200,True,456,0.802691,0.978488,0.797998,0.956175,21160
76,200,True,6854,0.847534,0.975880,0.818707,0.959590,21160
77,200,True,7580,0.798206,0.979791,0.799444,0.956744,21160
78,200,True,789,0.748879,0.966102,0.720503,0.938532,21160


In [11]:
# Define the metrics to analyze
metrics = ['class1_accuracy', 'class2_accuracy', 'kappa', 'global_accuracy', 'nfeat']

# Group results by number of superpixels and calculate mean and standard deviation for each metric
summary_stats = df_nsuper.groupby(['superpixel', 'filterReduction'])[metrics[:]].agg(['mean', 'std']).reset_index()

# Rename columns for easier access (e.g., 'kappa_mean', 'global_accuracy_mean')
summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns.values]

# # Select only the metrics of interest for visualization and highlight the highest values
# summary_stats = summary_stats[['superpixel_', 'filterReduction_', 'class1_accuracy_mean', 'class1_accuracy_std', 'class2_accuracy_mean', 'class2_accuracy_std', 'kappa_mean', 'kappa_std', 'global_accuracy_mean', 'global_accuracy_std']].style.highlight_max(
#     subset=['class1_accuracy_mean', 'class2_accuracy_mean', 'kappa_mean', 'global_accuracy_mean', 'global_accuracy_std'], color='gray'
# )

# # Format values as percentages for better presentation
# summary_stats = summary_stats.format({'kappa_mean': '{:.4%}', 'global_accuracy_mean': '{:.4%}'})

# Mostrar apenas algumas colunas específicas
# summary_stats[['superpixel_', 'filterReduction_', 'class1_accuracy_mean', 'class2_accuracy_mean', 'kappa_mean', 'global_accuracy_mean']]
summary_stats[['superpixel_', 'filterReduction_', 'class1_accuracy_mean', 'class2_accuracy_mean', 'kappa_mean', 'global_accuracy_mean']]

# latex
# latex_table = summary_stats.to_latex(float_format="%.6f")
# print(latex_table)

# # Renomeia as colunas para facilitar o acesso
# summary_stats_df.columns = ['superpixel', 'filterReduction'] + [f"{m}_{stat}" for m in metrics[:-1] for stat in ['mean', 'std']]

# # Ordena pelo número de superpixels e filterReduction
# summary_stats_df = summary_stats_df.sort_values(['superpixel', 'filterReduction']).reset_index(drop=True)

# summary_stats_df    

# # Destaca os maiores valores das métricas principais
# styled_summary = summary_stats_df.style.highlight_max(
#     subset=['class1_accuracy_mean', 'class2_accuracy_mean', 'kappa_mean', 'global_accuracy_mean', 'global_accuracy_std'],
#     color='gray'
# ).format({'kappa_mean': '{:.4%}', 'global_accuracy_mean': '{:.4%}'})

# styled_summary


# Display the styled DataFrame
# summary_stats

# Export the styled summary table to LaTeX code (without styling)
# latex_table = df_nsuper.groupby('superpixel')[metrics[:-1]].agg(['mean', 'std']).loc[superpixels_values].to_latex(float_format="%.4f")
# print(latex_table)

,superpixel_,filterReduction_,class1_accuracy_mean,class2_accuracy_mean,kappa_mean,global_accuracy_mean
0,50,False,0.869507,0.985202,0.865287,0.970518
1,50,True,0.772646,0.971382,0.753347,0.946158
2,100,False,0.878924,0.987158,0.878307,0.973421
3,100,True,0.781614,0.970274,0.756359,0.946329
4,150,False,0.879372,0.985398,0.872262,0.971941
5,150,True,0.787444,0.973859,0.772413,0.950199
6,200,False,0.867265,0.986180,0.867428,0.971087
7,200,True,0.777579,0.970078,0.753408,0.945646


In [6]:
# Wilcoxon signed-rank test para todos os pares de superpixel apenas para filterReduction_ = True
wilcoxon_results = pd.DataFrame(columns=['superpixel1', 'superpixel2', 'statistic', 'p_value'])

# Seleciona apenas os superpixels com filterReduction = True e ordena
superpixels_true = sorted(df_nsuper[df_nsuper['filterReduction'] == True]['superpixel'].unique())

for i in range(len(superpixels_true)):
    for j in range(i + 1, len(superpixels_true)):
        sp1 = superpixels_true[i]
        sp2 = superpixels_true[j]
        
        data_sp1 = df_nsuper[(df_nsuper['superpixel'] == sp1) & (df_nsuper['filterReduction'] == True)]['global_accuracy']
        data_sp2 = df_nsuper[(df_nsuper['superpixel'] == sp2) & (df_nsuper['filterReduction'] == True)]['global_accuracy']
        
        # Só faz o teste se os tamanhos das amostras forem iguais e maiores que 0
        if len(data_sp1) > 0 and len(data_sp2) > 0 and len(data_sp1) == len(data_sp2):
            statistic, p_value = wilcoxon(data_sp1, data_sp2)
            wilcoxon_results.loc[len(wilcoxon_results)] = {
                'superpixel1': sp1,
                'superpixel2': sp2,
                'statistic': statistic,
                'p_value': p_value
            }

print("Wilcoxon test results:")
print(wilcoxon_results)

significant = wilcoxon_results[wilcoxon_results['p_value'] < 0.05]
print("Comparisons with statistically significant difference:")
print(significant)
print(f"Total significant comparisons: {len(significant)} out of {len(wilcoxon_results)}")


Wilcoxon test results:
   superpixel1  superpixel2  statistic   p_value
0           50          100       27.5  1.000000
1           50          150       11.5  0.111328
2           50          200       26.0  0.921875
3          100          150       15.0  0.222656
4          100          200       16.0  0.843750
5          150          200        4.5  0.015625
Comparisons with statistically significant difference:
   superpixel1  superpixel2  statistic   p_value
5          150          200        4.5  0.015625
Total significant comparisons: 1 out of 6


In [7]:
# Wilcoxon signed-rank test para todos os pares de superpixel apenas para filterReduction_ = True
wilcoxon_results = pd.DataFrame(columns=['superpixel1', 'superpixel2', 'statistic', 'p_value'])

# Seleciona apenas os superpixels com filterReduction = True e ordena
superpixels_true = sorted(df_nsuper[df_nsuper['filterReduction'] == False]['superpixel'].unique())

for i in range(len(superpixels_true)):
    for j in range(i + 1, len(superpixels_true)):
        sp1 = superpixels_true[i]
        sp2 = superpixels_true[j]

        data_sp1 = df_nsuper[(df_nsuper['superpixel'] == sp1) & (df_nsuper['filterReduction'] == False)]['global_accuracy']
        data_sp2 = df_nsuper[(df_nsuper['superpixel'] == sp2) & (df_nsuper['filterReduction'] == False)]['global_accuracy']

        # Só faz o teste se os tamanhos das amostras forem iguais e maiores que 0
        if len(data_sp1) > 0 and len(data_sp2) > 0 and len(data_sp1) == len(data_sp2):
            statistic, p_value = wilcoxon(data_sp1, data_sp2)
            wilcoxon_results.loc[len(wilcoxon_results)] = {
                'superpixel1': sp1,
                'superpixel2': sp2,
                'statistic': statistic,
                'p_value': p_value
            }

print("Wilcoxon test results:")
print(wilcoxon_results)

significant = wilcoxon_results[wilcoxon_results['p_value'] < 0.05]
print("Comparisons with statistically significant difference:")
print(significant)
print(f"Total significant comparisons: {len(significant)} out of {len(wilcoxon_results)}")


Wilcoxon test results:
   superpixel1  superpixel2  statistic   p_value
0           50          100        8.0  0.046875
1           50          150       19.5  0.449219
2           50          200       15.5  0.445312
3          100          150       10.5  0.175781
4          100          200        9.0  0.128906
5          150          200       27.0  1.000000
Comparisons with statistically significant difference:
   superpixel1  superpixel2  statistic   p_value
0           50          100        8.0  0.046875
Total significant comparisons: 1 out of 6


In [12]:
# Path to the experiments directory
dir_experiments = "../experiments/nsuperpixels_filterReduction/results"

# Dicionários separados
results_files = defaultdict(dict)
superpixel_images = defaultdict(dict)

# Traverse the experiments directory
for superpixel_folder in sorted(os.listdir(dir_experiments)):
    superpixel_path = os.path.join(dir_experiments, superpixel_folder)
    if os.path.isdir(superpixel_path) and superpixel_folder.startswith("super"):
        # Armazena arquivos de resultados
        for file in sorted(os.listdir(superpixel_path)):
            if file.endswith("layer3_test1-classified-images.cvs"):
                results_files[superpixel_folder][file] = os.path.join(superpixel_path, file)
        # Armazena imagens dos superpixels
        for seed_folder in sorted(os.listdir(superpixel_path)):
            seed_path = os.path.join(superpixel_path, seed_folder)
            if os.path.isdir(seed_path) and seed_folder.startswith("superpixels_seed"):
                superpixel_images[superpixel_folder][seed_folder] = []
                for img_file in sorted(os.listdir(seed_path)):
                    superpixel_images[superpixel_folder][seed_folder].append(os.path.join(seed_path, img_file))

In [14]:
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# # montar matriz de confusão para cada nimage e semente
# confusion_matrices_nimage = {}
# for nimage, seeds_data in results_files_pred_nimages.items():
#     confusion_matrices_nimage[nimage] = {}
#     for seed, data in seeds_data.items():
#         correct = np.array(data['correct'])
#         predicted = np.array(data['predicted'])
#         cm = pd.crosstab(correct, predicted, rownames=['Actual'], colnames=['Predicted'], dropna=False)
#         confusion_matrices_nimage[nimage][seed] = cm

# evaluation_metrics_nimage = []
# for nimage, seeds_data in results_files_pred_nimages.items():
#     for seed, data in seeds_data.items():
#         y_true = data['correct']
#         y_pred = data['predicted']
#         cm = confusion_matrix(y_true, y_pred, labels=[1, 2])
#         tn, fp, fn, tp = cm.ravel()
#         accuracy = accuracy_score(y_true, y_pred)
#         precision = precision_score(y_true, y_pred, average='macro')
#         recall = recall_score(y_true, y_pred, average='macro')
#         f1 = f1_score(y_true, y_pred, average='macro')
#         evaluation_metrics_nimage.append({
#             "nimage": nimage,
#             "seed": seed,
#             "accuracy": accuracy,
#             "precision": precision,
#             "recall": recall,
#             "f1": f1
#         })

# df_metrics_nimage = pd.DataFrame(evaluation_metrics_nimage)
# df_metrics_nimage_mean = df_metrics_nimage.groupby('nimage').mean().reset_index()
# df_metrics_nimage_mean

# # mostrar codigo latex
# latex_table_nimage = df_metrics_nimage_mean.to_latex(float_format="%.6f")
# print(latex_table_nimage)